In [20]:
import os
import shutil
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [21]:
source_dir = "PlantVillage"
train_dir = "data_split/train"
val_dir = "data_split/val"

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

for class_name in os.listdir(source_dir):
    class_path = os.path.join(source_dir, class_name)

    # bỏ qua nếu không phải folder
    if not os.path.isdir(class_path):
        continue

    # chỉ lấy file ảnh
    images = [img for img in os.listdir(class_path)
              if img.lower().endswith(('.jpg', '.png', '.jpeg'))]

    # bỏ qua folder rác (như PlantVillage bên trong)
    if len(images) == 0:
        continue

    random.shuffle(images)
    split = int(0.8 * len(images))

    train_imgs = images[:split]
    val_imgs = images[split:]

    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

    for img in train_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(train_dir, class_name, img))

    for img in val_imgs:
        shutil.copy(os.path.join(class_path, img),
                    os.path.join(val_dir, class_name, img))

print("✅ DONE SPLIT")

✅ DONE SPLIT


In [22]:
train_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(25),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor()
])

val_transforms = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [23]:
train_dataset = datasets.ImageFolder("data_split/train", transform=train_transforms)
val_dataset = datasets.ImageFolder("data_split/val", transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_classes = len(train_dataset.classes)
print("Classes:", num_classes)

Classes: 15


In [24]:
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [26]:
def train_model(model, train_loader, val_loader, epochs=40):
    for epoch in range(epochs):
        model.train()
        train_correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()

        train_acc = train_correct / len(train_loader.dataset)

        # validation
        model.eval()
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)

                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()

        val_acc = val_correct / len(val_loader.dataset)

        print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

In [27]:
train_model(model, train_loader, val_loader, epochs=40)

Epoch 1: Train=0.5867, Val=0.7787
Epoch 2: Train=0.7575, Val=0.8241
Epoch 3: Train=0.8157, Val=0.8945
Epoch 4: Train=0.8547, Val=0.8778
Epoch 5: Train=0.8747, Val=0.9294
Epoch 6: Train=0.8907, Val=0.9349
Epoch 7: Train=0.8980, Val=0.9557
Epoch 8: Train=0.9088, Val=0.9405
Epoch 9: Train=0.9140, Val=0.9516
Epoch 10: Train=0.9183, Val=0.9565
Epoch 11: Train=0.9251, Val=0.9180
Epoch 12: Train=0.9343, Val=0.9698
Epoch 13: Train=0.9347, Val=0.9613
Epoch 14: Train=0.9383, Val=0.9635
Epoch 15: Train=0.9395, Val=0.9497
Epoch 16: Train=0.9432, Val=0.9613
Epoch 17: Train=0.9429, Val=0.9715
Epoch 18: Train=0.9466, Val=0.9666
Epoch 19: Train=0.9502, Val=0.9761
Epoch 20: Train=0.9544, Val=0.9741
Epoch 21: Train=0.9540, Val=0.9748
Epoch 22: Train=0.9556, Val=0.9724
Epoch 23: Train=0.9536, Val=0.9664
Epoch 24: Train=0.9604, Val=0.9775
Epoch 25: Train=0.9601, Val=0.9809
Epoch 26: Train=0.9618, Val=0.9763
Epoch 27: Train=0.9631, Val=0.9763
Epoch 28: Train=0.9644, Val=0.9727
Epoch 29: Train=0.9643, Val=0